# 第13章 RAG

## 13.2 基本的なRAGのシステムの実装

### LangChainでLLMと文埋め込みモデルを使う

#### 環境の準備

In [1]:
!pip install transformers[torch,sentencepiece] langchain==0.2.12 langchain-community==0.2.11 langchain-huggingface==0.0.3 faiss-cpu jq bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.2 MB/s eta 0:00:00


In [2]:
from huggingface_hub import notebook_login

# Hugging Face Hubにログイン
notebook_login()

In [3]:
from transformers.trainer_utils import set_seed
set_seed(42)

#### LangChainでLLMを使う

In [4]:
import torch
from langchain_huggingface import HuggingFacePipeline
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    pipeline,
    BitsAndBytesConfig
)

# HuggingFaceHubにおけるモデル名を指定
model_name = "llm-book/Swallow-7b-hf-oasst1-21k-ja"

# 8bit量子化の設定
quantization_config = BitsAndBytesConfig(
    load_in_8bit=True
)

# モデルを読み込む（8bit量子化を適用）
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto",
)

# トークナイザを読み込む
tokenizer = AutoTokenizer.from_pretrained(model_name)

# テキスト生成用のパラメータを指定
generation_config = {
    "max_new_tokens": 128,
    "do_sample": False,
    "temperature": None,
    "top_p": None,
}

# テキスト生成を行うパイプラインを作成
text_generation_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device_map="auto",
    **generation_config
)

# パイプラインからLangChainのLLMコンポーネントを作成
llm = HuggingFacePipeline(pipeline=text_generation_pipeline)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'top_p', 'temperature', 'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [5]:
from pprint import pprint

# modelに入力する会話データ
llm_prompt_messages = [
    {"role": "user", "content": "四国地方で一番高い山は？"},
]

# 会話データにチャットテンプレートを適用し、内容を確認
llm_prompt_text = tokenizer.apply_chat_template(
    llm_prompt_messages,
    tokenize=False,
    add_generation_prompt=True,
)
pprint(llm_prompt_text)


'<s>ユーザ：四国地方で一番高い山は？</s><s>アシスタント：'


In [6]:
# LLMへの入力を実行し結果を確認
llm_output_message = llm.invoke(llm_prompt_text)
pprint(llm_output_message)

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Both `max_new_tokens` (=128) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer LlamaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this

'<s>ユーザ：四国地方で一番高い山は？</s><s>アシスタント：四国地方で一番高い山は、徳島県と高知県の県境にある剣山（つるぎさん、1,955m）である。'


#### Chat Modelコンポーネントの利用

In [7]:
from langchain_huggingface import ChatHuggingFace

# LLMコンポーネントからChatModelコンポーネントを作成
chat_model = ChatHuggingFace(llm=llm, tokenizer=tokenizer)

In [8]:
from langchain_core.messages import HumanMessage, SystemMessage

# Chat Modelに入力する会話データ
chat_messages = [HumanMessage(content="四国地方で一番高い山は？")]
#print(chat_messages)

# Chat Modelによるチャットテンプレート適用後の入力文字列をかくにん
chat_prompt = chat_model._to_chat_prompt(chat_messages)
print(chat_prompt)

<s>ユーザ：四国地方で一番高い山は？</s><s>アシスタント：


In [9]:
# Chat Modelに会話データを入力し、出力を確認
chat_output_message = chat_model.invoke(chat_messages)
pprint(chat_output_message)

[transformers] Both `max_new_tokens` (=128) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


AIMessage(content='<s>ユーザ：四国地方で一番高い山は？</s><s>アシスタント：四国地方で一番高い山は、徳島県と高知県の県境にある剣山（つるぎさん、1,955m）である。', id='run-d6de1634-71a9-4a52-a6e4-0042e6c0e7ab-0')


In [10]:
# Chat Modelが出力したテキストからモデルの応答部分のみを抽出
response_text = chat_output_message.content[len(chat_prompt) :]
print(response_text)

四国地方で一番高い山は、徳島県と高知県の県境にある剣山（つるぎさん、1,955m）である。


#### Chainを構築する

In [11]:
from langchain_core.prompts import ChatPromptTemplate

# 任意のqueryからプロンプトを構築するPrompt Templateを作成
prompt_template = ChatPromptTemplate.from_messages(
    [("user", "{query}")]
)

# Prompt Templateを実行し、結果を確認
prompt_template_output = prompt_template.invoke(
    {"query": "四国地方で一番高い山は？"}
)
pprint(prompt_template_output)

ChatPromptValue(messages=[HumanMessage(content='四国地方で一番高い山は？')])


In [12]:
# Prompt TemplateとChat Modelを連結したChainを作成
chain = prompt_template | chat_model
print(chain)

# Chainをジック押し結果を確認
chain_output = chain.invoke({"query": "四国地方で一番高い山は？"})
pprint(chain_output)

[transformers] Both `max_new_tokens` (=128) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


first=ChatPromptTemplate(input_variables=['query'], messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['query'], template='{query}'))]) last=ChatHuggingFace(llm=HuggingFacePipeline(pipeline=TextGenerationPipeline: {'model': 'LlamaForCausalLM', 'dtype': 'bfloat16', 'device': 'cuda', 'input_modalities': 'text', 'output_modalities': ('text',)}), tokenizer=LlamaTokenizer(name_or_path='llm-book/Swallow-7b-hf-oasst1-21k-ja', vocab_size=43176, model_max_length=1000000000000000019884624838656, padding_side='left', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '<unk>'}, added_tokens_decoder={
	0: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	1: AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
}), mod

/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


AIMessage(content='<s>ユーザ：四国地方で一番高い山は？</s><s>アシスタント：四国地方で一番高い山は、徳島県と高知県の県境にある剣山（つるぎさん、1,955m）である。', id='run-755fa4bf-8552-457f-a27f-05614f6a46e9-0')


In [13]:
from langchain_core.prompt_values import ChatPromptValue
from langchain_core.runnables import RunnableLambda

def chat_model_resp_func(
    chat_prompt_value: ChatPromptValue,
) -> str:
    """
    chat_modelにchat_prompt_valueを入力し
    出力からモデルの応答部分のみを文字列で返す
    """
    chat_prompt = chat_model._to_chat_prompt(
        chat_prompt_value.messages
    )
    chat_output_message = chat_model.invoke(chat_prompt_value)
    response_text = chat_output_message.content[len(chat_prompt) :]
    return response_text

# 定義した関数の処理を行うRunnableを作成
chat_model_resp_only = RunnableLambda(chat_model_resp_func)

# prompt templateとRunnableを連結したChainを作成
chain_resp_only = prompt_template | chat_model_resp_only

# Chainを実行し結果を確認
chain_resp_only_output = chain_resp_only.invoke(
    {"query": "四国地方で一番高い山は？"}
)
print(chain_resp_only_output)

[transformers] Both `max_new_tokens` (=128) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


四国地方で一番高い山は、徳島県と高知県の県境にある剣山（つるぎさん、1,955m）である。


#### LangChainで文埋め込みモデルを使う

In [14]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

# Hugging Face Hubにおけるモデル名を指定
embedding_model_name = "BAAI/bge-m3"

# モデル名からEmbedding Modelを初期化
embedding_model = HuggingFaceEmbeddings(
    model_name=embedding_model_name,
    model_kwargs={"model_kwargs": {"torch_dtype": torch.float16}},
)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [15]:
sample_texts = [
    "日本で一番高い山は何ですか？",
    "日本で一番高い山は富士山です。",
]

# 2つのテキストに対して文埋め込みを実行し、結果を確認
sample_embeddings = embedding_model.embed_documents(sample_texts)
print(sample_embeddings)

[[0.010528564453125, 0.032501220703125, -0.0242767333984375, -0.025299072265625, 0.0111541748046875, -0.0303802490234375, -0.01141357421875, -0.011871337890625, -0.034393310546875, -0.0261077880859375, -0.0206298828125, 0.0270843505859375, -0.0203704833984375, 0.007770538330078125, 0.042205810546875, -0.02752685546875, 0.042510986328125, 0.001697540283203125, -0.0020751953125, 0.023223876953125, -0.00534820556640625, -0.021270751953125, -0.0308074951171875, 0.0306396484375, 0.006214141845703125, 0.0177154541015625, -0.01160430908203125, 0.0045928955078125, -0.005519866943359375, -0.038055419921875, -0.023590087890625, -0.0204315185546875, 0.002490997314453125, -0.0026531219482421875, -0.028106689453125, -0.0082855224609375, -0.00017535686492919922, -0.0306396484375, -0.03692626953125, 0.0010480880737304688, 0.034088134765625, 0.019561767578125, 0.04058837890625, -0.024322509765625, -0.0438232421875, -0.0181121826171875, -0.01922607421875, -0.0309295654296875, 0.018829345703125, -0.0056

In [16]:
# 2つのテキストの文埋め込みから類似度を計算
similarity = torch.nn.functional.cosine_similarity(
    torch.tensor([sample_embeddings[0]]),
    torch.tensor([sample_embeddings[1]]),
)
print(similarity)

tensor([0.7743])


### 13.2.3 LangChainでRAGの実装

#### データストアの構築

In [17]:
# 検索対象の文書集合のファイルをダウンロード
!wget \
https://github.com/ghmagazine/llm-book/raw/main/chapter13/docs.json

--2026-06-11 00:23:43--  https://github.com/ghmagazine/llm-book/raw/main/chapter13/docs.json
Resolving github.com (github.com)... 140.82.113.4
Connecting to github.com (github.com)|140.82.113.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/ghmagazine/llm-book/main/chapter13/docs.json [following]
--2026-06-11 00:23:43--  https://raw.githubusercontent.com/ghmagazine/llm-book/main/chapter13/docs.json
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1178382 (1.1M) [text/plain]
Saving to: ‘docs.json.3’

docs.json.3         100%[===================>]   1.12M  --.-KB/s    in 0.03s   

2026-06-11 00:23:47 (34.3 MB/s) - ‘docs.json.3’ saved [1178382/1178382]



In [18]:
from langchain_community.document_loaders import JSONLoader

# JSONファイルから文書を読み込むためのDocument Loaderを初期化
document_loader = JSONLoader(
    file_path ="./docs.json",
    jq_schema=".text",          # 読み込み対象のフィールド
    json_lines=True,            # JSON Lines形式のファイルであることを指定
)

# 文書の読み込みを実行
documents = document_loader.load()

# 読み込まれた文書数を確認
print(len(documents))

103


In [19]:
for i,doc in enumerate(documents):
    print(doc)
    if i > 10:
        break

page_content='富士山（ふじさん）は、静岡県（富士宮市、富士市、裾野市、御殿場市、駿東郡小山町）と山梨県（富士吉田市、南都留郡鳴沢村）に跨る活火山である。標高3776.12 m、日本最高峰（剣ヶ峰）の独立峰で、その優美な風貌は日本国外でも日本の象徴として広く知られている。 数多くの芸術作品の題材とされ芸術面のみならず、気候や地層など地質学的にも社会に大きな影響を与えている。懸垂曲線の山容を有した玄武岩質成層火山で構成され、その山体は駿河湾の海岸まで及ぶ。 古来より霊峰とされ、特に山頂部は浅間大神が鎮座するとされたため、神聖視された。噴火を沈静化するため律令国家により浅間神社が祭祀され、浅間信仰が確立された。また、富士山修験道の開祖とされる富士上人により修験道の霊場としても認識されるようになり、登拝が行われるようになった。これら富士信仰は時代により多様化し、村山修験や富士講といった一派を形成するに至る。現在、富士山麓周辺には観光名所が多くある他、夏季シーズンには富士登山が盛んである。 日本三名山（三霊山）、日本百名山、日本の地質百選に選定されている。また、1936年（昭和11年）には富士箱根伊豆国立公園に指定されている。その後、1952年（昭和27年）に特別名勝、2011年（平成23年）に史跡、さらに2013年（平成25年）6月22日には関連する文化財群とともに「富士山-信仰の対象と芸術の源泉」の名で世界文化遺産に登録された。 富士山についての最も古い記録は『常陸国風土記』における「福慈岳」という語であると言われている。他にも多くの呼称が存在し、不二山もしくは不尽山と表記する古文献もある。また、『竹取物語』における伝説もある。「フジ」という長い山の斜面を表す大和言葉から転じて富士山と称されたという説もある。近代以降の語源説としては、宣教師バチェラーは、名前は「火を噴く山」を意味するアイヌ語の「フンチヌプリ」に由来するとの説を提示した。しかし、これは囲炉裏の中に鎮座する火の姥神を表す「アペフチカムイ」からきた誤解であるとの反論がある。その他の語源説として、マレー語説、マオリ語説、原ポリネシア語説がある。 明確に「富士山」と表記される過程においては駿河国に由来するとするものがあり、記録としては都良香の『富士山記』に「山を富士と名づくるは、郡の名に取れるなり」と

In [20]:
print(documents[0].type)

Document


In [21]:
print(len(documents[0].page_content))

21232


In [22]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 文書を指定した文字数で分割するSplitterを初期化
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,         # 分割する最大文字数
    chunk_overlap=100,      # 分割された文書間で重複させる最大文字数
    add_start_index=True,   # 元の文書における開始位置の情報を付与
)

# 文書の分割を実行
split_documents = text_splitter.split_documents(documents)

# 分割後の文書数を確認
print(len(split_documents))

1475


In [23]:
print(split_documents[0])
print(split_documents[1])

page_content='富士山（ふじさん）は、静岡県（富士宮市、富士市、裾野市、御殿場市、駿東郡小山町）と山梨県（富士吉田市、南都留郡鳴沢村）に跨る活火山である。標高3776.12 m、日本最高峰（剣ヶ峰）の独立峰で、その優美な風貌は日本国外でも日本の象徴として広く知られている。 数多くの芸術作品の題材とされ芸術面のみならず、気候や地層など地質学的にも社会に大きな影響を与えている。懸垂曲線の山容を有した玄武岩質成層火山で構成され、その山体は駿河湾の海岸まで及ぶ。' metadata={'source': '/content/docs.json', 'seq_num': 1, 'start_index': 0}
page_content='数多くの芸術作品の題材とされ芸術面のみならず、気候や地層など地質学的にも社会に大きな影響を与えている。懸垂曲線の山容を有した玄武岩質成層火山で構成され、その山体は駿河湾の海岸まで及ぶ。 古来より霊峰とされ、特に山頂部は浅間大神が鎮座するとされたため、神聖視された。噴火を沈静化するため律令国家により浅間神社が祭祀され、浅間信仰が確立された。また、富士山修験道の開祖とされる富士上人により修験道の霊場としても認識されるようになり、登拝が行われるようになった。これら富士信仰は時代により多様化し、村山修験や富士講といった一派を形成するに至る。現在、富士山麓周辺には観光名所が多くある他、夏季シーズンには富士登山が盛んである。' metadata={'source': '/content/docs.json', 'seq_num': 1, 'start_index': 129}


In [24]:
# 分割後の文書の長さ（文字数）を確認
print(len(split_documents[0].page_content))
print(len(split_documents[1].page_content))

221
310


In [25]:
print(split_documents[0].page_content[-100:])
print(split_documents[1].page_content[0:100])

知られている。 数多くの芸術作品の題材とされ芸術面のみならず、気候や地層など地質学的にも社会に大きな影響を与えている。懸垂曲線の山容を有した玄武岩質成層火山で構成され、その山体は駿河湾の海岸まで及ぶ。
数多くの芸術作品の題材とされ芸術面のみならず、気候や地層など地質学的にも社会に大きな影響を与えている。懸垂曲線の山容を有した玄武岩質成層火山で構成され、その山体は駿河湾の海岸まで及ぶ。 古来より霊峰と


#### ベクトルインデックスの作成

In [26]:
from langchain_community.vectorstores import FAISS

# 分割後の文書と埋め込みモデルを用いて、Faissのベクトルインデックスを作成
vectorstore = FAISS.from_documents(split_documents, embedding_model)

# ベクトルインデックスに登録された文書数を確認
print(vectorstore.index.ntotal)

1475


In [27]:
print(vectorstore.index)

<faiss.swigfaiss.IndexFlatL2; proxy of <Swig Object of type 'faiss::IndexFlatL2 *' at 0x7a7e457b7c30> >


In [28]:
print(vectorstore.index.d)               # ベクトルの次元数
print(vectorstore.index_to_docstore_id)  # {0: 'uuid...', 1: 'uuid...'}
print(vectorstore.docstore._dict)        # {'uuid...': Document(...), ...}

# 生ベクトルそのものを取り出す
import numpy as np
vec = vectorstore.index.reconstruct(0)  # 0番目のベクトル(np.ndarray)

1024
{0: '3bfc2799-9423-4cbd-ab89-71dbf3ebe63e', 1: '5b48e21e-1f9f-439b-b725-42b022771c36', 2: '0d4ead80-6f40-4d22-b5d2-44beed600c6c', 3: '480af680-7eef-488b-9185-51aab7dff434', 4: 'cb3e558c-235d-4f28-adec-24327eafa5ec', 5: '255c80b8-7356-408f-8a70-f7b6aa8d433c', 6: 'b1fc89e5-6e90-4dcb-b2ba-003446e868ce', 7: '00de3727-205e-4618-ac86-ab14810c8f04', 8: '1e8bcf12-afeb-4ebd-b702-7e810fa4202d', 9: 'f334576e-9462-4eac-b9ff-9773f0898875', 10: '522ffa03-29ad-42b5-8b07-2f64cbbe58f4', 11: '0be2ae37-b29d-4314-89fd-2aefe19521cd', 12: 'e0dc9ede-5527-4267-ab2c-0aa7cf09fa90', 13: 'cea48141-dc79-4301-ba0b-bd2548eb87a5', 14: 'efd1b0be-9767-44bd-9ec1-eada66c62199', 15: '742d160a-b3c3-49f9-9f6d-35092fc7fc3e', 16: '992d04fa-8c1a-4742-b256-55f5725d4620', 17: 'da47302e-4423-4d1c-97cd-51321b818600', 18: 'da825f3f-218f-4370-85cc-c579ba26e9f5', 19: 'fe29b6a4-2832-44d0-8ebb-1bcedad79fa9', 20: '05ef9ed6-b747-482c-b177-ae68bebb8814', 21: '26c051b0-4374-41ce-b5b4-4661b469d47f', 22: '67a47730-08b2-400e-8b7e-acd2655

In [29]:
print(vec)

[ 0.03143311 -0.0019331  -0.001688   ... -0.02029419  0.00391388
  0.05728149]


In [30]:
print(vectorstore.docstore._dict.keys())

dict_keys(['3bfc2799-9423-4cbd-ab89-71dbf3ebe63e', '5b48e21e-1f9f-439b-b725-42b022771c36', '0d4ead80-6f40-4d22-b5d2-44beed600c6c', '480af680-7eef-488b-9185-51aab7dff434', 'cb3e558c-235d-4f28-adec-24327eafa5ec', '255c80b8-7356-408f-8a70-f7b6aa8d433c', 'b1fc89e5-6e90-4dcb-b2ba-003446e868ce', '00de3727-205e-4618-ac86-ab14810c8f04', '1e8bcf12-afeb-4ebd-b702-7e810fa4202d', 'f334576e-9462-4eac-b9ff-9773f0898875', '522ffa03-29ad-42b5-8b07-2f64cbbe58f4', '0be2ae37-b29d-4314-89fd-2aefe19521cd', 'e0dc9ede-5527-4267-ab2c-0aa7cf09fa90', 'cea48141-dc79-4301-ba0b-bd2548eb87a5', 'efd1b0be-9767-44bd-9ec1-eada66c62199', '742d160a-b3c3-49f9-9f6d-35092fc7fc3e', '992d04fa-8c1a-4742-b256-55f5725d4620', 'da47302e-4423-4d1c-97cd-51321b818600', 'da825f3f-218f-4370-85cc-c579ba26e9f5', 'fe29b6a4-2832-44d0-8ebb-1bcedad79fa9', '05ef9ed6-b747-482c-b177-ae68bebb8814', '26c051b0-4374-41ce-b5b4-4661b469d47f', '67a47730-08b2-400e-8b7e-acd26553a5c2', 'cc7c5466-52b9-41b5-ada3-15cf43b05f61', '7f5437ea-075a-46da-bf55-226a

In [31]:
print(vectorstore.docstore._dict.values())

dict_values([Document(metadata={'source': '/content/docs.json', 'seq_num': 1, 'start_index': 0}, page_content='富士山（ふじさん）は、静岡県（富士宮市、富士市、裾野市、御殿場市、駿東郡小山町）と山梨県（富士吉田市、南都留郡鳴沢村）に跨る活火山である。標高3776.12 m、日本最高峰（剣ヶ峰）の独立峰で、その優美な風貌は日本国外でも日本の象徴として広く知られている。 数多くの芸術作品の題材とされ芸術面のみならず、気候や地層など地質学的にも社会に大きな影響を与えている。懸垂曲線の山容を有した玄武岩質成層火山で構成され、その山体は駿河湾の海岸まで及ぶ。'), Document(metadata={'source': '/content/docs.json', 'seq_num': 1, 'start_index': 129}, page_content='数多くの芸術作品の題材とされ芸術面のみならず、気候や地層など地質学的にも社会に大きな影響を与えている。懸垂曲線の山容を有した玄武岩質成層火山で構成され、その山体は駿河湾の海岸まで及ぶ。 古来より霊峰とされ、特に山頂部は浅間大神が鎮座するとされたため、神聖視された。噴火を沈静化するため律令国家により浅間神社が祭祀され、浅間信仰が確立された。また、富士山修験道の開祖とされる富士上人により修験道の霊場としても認識されるようになり、登拝が行われるようになった。これら富士信仰は時代により多様化し、村山修験や富士講といった一派を形成するに至る。現在、富士山麓周辺には観光名所が多くある他、夏季シーズンには富士登山が盛んである。'), Document(metadata={'source': '/content/docs.json', 'seq_num': 1, 'start_index': 440}, page_content='日本三名山（三霊山）、日本百名山、日本の地質百選に選定されている。また、1936年（昭和11年）には富士箱根伊豆国立公園に指定されている。その後、1952年（昭和27年）に特別名勝、2011年（平成23年）に史跡、さらに2013年（平成25年）6月22日には関連する文化財群とともに「富士山-信仰の対

#### Retrieverコンポーネントの作成

In [32]:
# ベクトルインデックスを元に文書の検索を行うRetrieverを初期化
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

In [33]:
# 文書の検索を実行
retriever_documents = retriever.invoke("四国地方で一番高い山は？")

pprint(retriever_documents)

[Document(metadata={'source': '/content/docs.json', 'seq_num': 26, 'start_index': 0}, page_content='この項目に含まれる文字「鎚」は、オペレーティングシステムやブラウザなどの環境により表示が異なります。 石鎚山（いしづちさん、いしづちやま）は、四国山地西部に位置する標高1,982 mの山で、近畿以西を「西日本」とした場合の西日本最高峰で、山頂から望む展望が四国八十八景64番に選定。愛媛県西条市と久万高原町の境界に位置する。 石鉄山、石鈇山、石土山、石槌山とも表記され、伊予の高嶺とも呼ばれる。『日本霊異記』には「石槌山」と記され、延喜式の神名帳（延喜式神名帳）では「石鉄神社」と記されている。前神寺および横峰寺では「石鈇山（しゃくまざん）」とも呼ぶ。'),
 Document(metadata={'source': '/content/docs.json', 'seq_num': 1, 'start_index': 0}, page_content='富士山（ふじさん）は、静岡県（富士宮市、富士市、裾野市、御殿場市、駿東郡小山町）と山梨県（富士吉田市、南都留郡鳴沢村）に跨る活火山である。標高3776.12 m、日本最高峰（剣ヶ峰）の独立峰で、その優美な風貌は日本国外でも日本の象徴として広く知られている。 数多くの芸術作品の題材とされ芸術面のみならず、気候や地層など地質学的にも社会に大きな影響を与えている。懸垂曲線の山容を有した玄武岩質成層火山で構成され、その山体は駿河湾の海岸まで及ぶ。')]


In [34]:
test_retriever_doc = retriever.invoke("日本の神は？")
print(test_retriever_doc[0].page_content)

両神山（りょうかみさん）は埼玉県秩父郡小鹿野町と秩父市の境目にある山。奥秩父山塊の北部にあり、標高は1,723 m。日本百名山の一つ。山岳信仰の霊峰であり、両神山、三峰山、武甲山をあわせて「秩父三山」という。 山名は、イザナギ、イザナミの神を祀っていることから両神と呼ぶという説、日本武尊の東征のおりこの山を八日間見ながら通過していったので八日見山と名づけられた説、「龍神を祭る山」が転じて両神山となったという説など、諸説ある。


#### RAGのChainの構築

In [35]:
# 任意のqueryからメッセージを構築するPrompt Templateを作成
rag_prompt_text = (
    "以下の文書の内容を参考にして、質問に答えてください。\n\n"
    "---\n{context}\n---\n\n質問: {query}"
)
rag_prompt_template = ChatPromptTemplate.from_messages(
    [("user", rag_prompt_text)]
)

In [36]:
from langchain_core.documents import Document

def format_documents_func(documents: list[Document]) -> str:
    """文書のリストを改行で連結した一つの文字列として返す"""
    return "\n\n".join(
        document.page_content for document in documents
    )

format_documents = RunnableLambda(format_documents_func)

In [37]:
from langchain_core.runnables import RunnablePassthrough

# RAGの一連の処理を行うChainを作成
rag_chain = (
    {
        "context": retriever | format_documents,
        "query": RunnablePassthrough(),
    }
    | rag_prompt_template
    | chat_model_resp_only
)

In [38]:
rag_chain_output = rag_chain.invoke("四国地方で一番高い山は？")
print(rag_chain_output)

[transformers] Both `max_new_tokens` (=128) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


四国地方で一番高い山は、愛媛県と高知県にまたがる石鎚山で、標高は1,982メートルです。
